# Weakly Nonlinear Shear-Thinning Drop: Symbolic Derivation

This notebook derives the perturbation expansion for a shear-thinning drop oscillating on a flat substrate, using the Carreau constitutive law. It follows the same non-dimensionalisation as the Newtonian (Reid 1960) and Oldroyd-B analyses in the companion documents.

**Structure:**  
Each section derives one piece of the chain symbolically, then asserts the result against an independent check. A failing assertion means the derivation broke — fix the math, not the assertion.

**Notation** (matches `reid1960_expanded-3.tex`):  
- $x = r/R$, dimensionless radial coordinate  
- $\epsilon$ = oscillation amplitude, small parameter  
- $\sigma$ = complex decay rate ($\operatorname{Re}(\sigma) > 0$ = decay)  
- $q^2 = \sigma R^2 / \nu$, viscous wavenumber  
- $\alpha^2 = \sigma_{l;0} R^2 / \nu$, inviscid frequency scaled  
- $U(x)$: radial velocity eigenfunction from Reid  
- $b_l(t)$: dimensionless mode amplitude (what the solver calls $A_l$)  
- $\mathrm{Oh} = \nu\sqrt{\rho/(\sigma_s R)}$: Ohnesorge number  

In [ ]:
import sympy as sp
from sympy import (
    symbols, Function, sqrt, Rational, simplify, expand, factor,
    diff, integrate, cos, sin, pi, I, oo, exp, conjugate,
    legendre, assoc_legendre, besselj, hankel1,
    series, limit, latex, Eq, solve, Symbol, lambdify,
    trigsimp, radsimp, cancel, apart, collect
)
from sympy.abc import x, r, n

sp.init_printing(use_unicode=True)

---
## 1. Carreau Constitutive Law and Perturbation Expansion

The Carreau model relates the effective viscosity to the scalar shear rate $\dot\gamma = \sqrt{2 e_{ij} e_{ij}}$:

$$\mu_{\mathrm{eff}}(\dot\gamma) = \mu_0 \left[1 + (\lambda_c \dot\gamma)^2\right]^{(n-1)/2}$$

with $n \in (0, 1]$ the power-law index ($n=1$: Newtonian) and $\lambda_c$ the Carreau relaxation time.

The oscillation amplitude provides a natural small parameter: $\dot\gamma = O(\epsilon)$ in the linearised flow.
We expand $\mu_{\mathrm{eff}}$ in $\epsilon$ by writing $\dot\gamma = \epsilon\, \hat{\dot\gamma}$.

In [ ]:
mu0, lam_c, eps, gdot_hat = symbols('mu_0 lambda_c epsilon hat_gamma', positive=True, real=True)
n_idx = symbols('n', positive=True, real=True)   # power-law index

gdot = eps * gdot_hat   # full shear rate

mu_carreau = mu0 * (1 + (lam_c * gdot)**2) ** ((n_idx - 1) / 2)

# Expand to O(eps^4)
mu_expanded = series(mu_carreau, eps, 0, 5)
print("mu_eff expanded in epsilon:")
mu_expanded

In [ ]:
# Extract correction terms by order
mu_O1   = mu_expanded.coeff(eps, 0)   # leading order (constant)
mu_corr2 = mu_expanded.coeff(eps, 2)  # O(eps^2) correction
mu_corr4 = mu_expanded.coeff(eps, 4)  # O(eps^4) correction

print("O(eps^0):", mu_O1)
print("O(eps^1):", mu_expanded.coeff(eps, 1))   # should be 0
print("O(eps^2):", mu_corr2)
print("O(eps^3):", mu_expanded.coeff(eps, 3))   # should be 0

In [ ]:
# ASSERTION 1: No O(eps^1) or O(eps^3) terms — the expansion is in even powers only.
assert mu_expanded.coeff(eps, 1) == 0, "O(eps^1) term should vanish"
assert mu_expanded.coeff(eps, 3) == 0, "O(eps^3) term should vanish"
print("PASS: viscosity correction has only even powers of epsilon")

# ASSERTION 2: At n=1 (Newtonian), all corrections vanish.
assert simplify(mu_corr2.subs(n_idx, 1)) == 0, "Newtonian limit: O(eps^2) correction should vanish"
print("PASS: n=1 recovers Newtonian (zero correction)")

# ASSERTION 3: At lambda_c=0, all corrections vanish.
assert simplify(mu_corr2.subs(lam_c, 0)) == 0, "lambda_c=0: correction should vanish"
print("PASS: lambda_c=0 recovers Newtonian")

### Key result: the viscosity correction

$$\mu_{\mathrm{eff}} = \mu_0 \left[1 + \underbrace{\frac{n-1}{2}(\lambda_c \dot\gamma)^2}_{O(\epsilon^2)} + O(\dot\gamma^4)\right]$$

**The $O(\epsilon^1)$ viscosity correction is exactly zero.** The first-order problem is purely Newtonian with $\mu_0$. Shear-thinning first enters at $O(\epsilon^2)$ in the viscosity, which produces an $O(\epsilon^3)$ correction to the stress (one extra factor of $\epsilon$ from $e_{ij}$).

We define the shear-thinning parameter $\varepsilon_{ST} = (1-n)/2 \geq 0$.

In [ ]:
eps_ST = (1 - n_idx) / 2

# The viscosity correction at O(eps^2) in terms of eps_ST
mu_corr2_clean = simplify(mu_corr2.subs((n_idx-1)/2, -eps_ST))

print("mu_corr2 =", mu_corr2)
print("In terms of eps_ST: mu_eff ≈ mu0 * [1 - eps_ST * (lambda_c * gdot_hat)^2 * eps^2 + ...]")
print("(n-1)/2 =", simplify((n_idx - 1)/2), "= -eps_ST")

---
## 2. Order $\epsilon^1$ Problem: Reid Velocity Field

At $O(\epsilon)$, the flow is Newtonian with viscosity $\mu_0$. The radial velocity eigenfunction $U(x)$ satisfies Reid's ODE (eq. 24 in `reid1960_expanded-3.tex`):

$$\left[\frac{d^2}{dx^2} - \frac{l(l+1)}{x^2} + q^2\right]U(x) = q^2 \Pi_0\, x^{l+1}$$

with general solution $U(x) = C\, x\, j_l(qx) + \Pi_0\, x^{l+1}$,
where $j_l$ is the spherical Bessel function of the first kind.

The full radial velocity is $u_r = \dot{b}_l\, U(x)\, P_l(\cos\theta)$ (the $1/x^2$ factor from Reid absorbed into $U$ here for clarity).

In [ ]:
x_sym, q, l_sym = symbols('x q l', positive=True, real=True)
C_sym, Pi0 = symbols('C Pi_0')

# Spherical Bessel j_l(z) = sqrt(pi/(2z)) * J_{l+1/2}(z)
def sph_bessel_j(l_val, z):
    return sqrt(pi / (2*z)) * besselj(l_val + sp.Rational(1,2), z)

# General solution to Reid's ODE
def U_field(x_val, l_val, q_val, C_val, Pi0_val):
    return C_val * x_val * sph_bessel_j(l_val, q_val * x_val) + Pi0_val * x_val**(l_val + 1)

# Verify U satisfies the ODE for a concrete mode (l=2, symbolic q)
l_test = 2
U_expr = U_field(x_sym, l_test, q, C_sym, Pi0)

# Reid ODE: U'' - l(l+1)/x^2 * U + q^2 * U = q^2 * Pi0 * x^(l+1)
ODE_lhs = diff(U_expr, x_sym, 2) - l_test*(l_test+1)/x_sym**2 * U_expr + q**2 * U_expr
ODE_rhs = q**2 * Pi0 * x_sym**(l_test + 1)

residual = simplify(ODE_lhs - ODE_rhs)
print("ODE residual (should be 0):")
print(simplify(residual))

In [ ]:
# ASSERTION 4: U(x) satisfies Reid's ODE exactly.
assert simplify(residual) == 0, f"Reid ODE not satisfied. Residual: {residual}"
print("PASS: U(x) = C*x*j_l(qx) + Pi0*x^(l+1) satisfies Reid's ODE")

### Boundary conditions and solution for $C$, $\Pi_0$

From `reid1960_expanded-3.tex` §7:

- **BC1** (kinematic): $U(1) = -1$  
- **BC2** (zero tangential stress at free surface): $\left[U'' - \frac{2}{x}U' + \frac{l(l+1)}{x^2}U\right]_{x=1} = 0$

These two conditions fix $C$ and $\Pi_0$ in terms of $q$ and $l$.

In [ ]:
# Apply BCs symbolically for l=2
# Use symbolic j_l evaluated at q (x=1)
jl   = symbols('j_l',   real=True)   # j_l(q)
jlp  = symbols('j_lp',  real=True)   # j_l'(q)  = d/d(qx) j_l evaluated at x=1, times q
l_s  = symbols('l', positive=True, integer=True)

# U(1) = C*j_l(q) + Pi0 = -1   [BC1]
# U''(1) - 2*U'(1) + l(l+1)*U(1) = 0  [BC2]
#
# From Reid eq. (37), BC2 applied gives:
# C*[-q^2*j_l + 2(l^2+l-1)*j_l - 2*q*j_lp] + 2*(l^2-1)*Pi0 = 0
# where j_lp = d/dz j_l(z)|_{z=q}

# Solve the 2x2 system {BC1, BC2} for C and Pi0
A_mat = sp.Matrix([
    [jl,                                                    1            ],
    [-q**2*jl + 2*(l_s**2+l_s-1)*jl - 2*q*jlp,   2*(l_s**2-1) ]
])
b_vec = sp.Matrix([-1, 0])

sol = A_mat.solve(b_vec)
C_sol   = sol[0]
Pi0_sol = sol[1]

print("C =")
sp.pprint(C_sol)
print("\nPi0 =")
sp.pprint(Pi0_sol)

In [ ]:
# The standard Reid result (eq. 38) is:
# C = 2*(l-1)*(l+1) / [(2l - q^2)*j_l - 2q*j_l']
C_reid = 2*(l_s-1)*(l_s+1) / ((2*l_s - q**2)*jl - 2*q*jlp)

# ASSERTION 5: Our BC solution matches Reid eq. (38)
diff_C = simplify(C_sol - C_reid)
assert diff_C == 0, f"C solution does not match Reid eq. 38. Difference: {diff_C}"
print("PASS: BC solution for C matches Reid eq. (38)")

# ASSERTION 6: BC1 is satisfied by the solution
BC1_check = simplify(C_sol * jl + Pi0_sol + 1)
assert BC1_check == 0, f"BC1 not satisfied. Residual: {BC1_check}"
print("PASS: BC1 satisfied: U(1) = -1")

---
## 3. Velocity Field in Full

With $U(x)$ determined, the velocity components for mode $l$ are (using incompressibility):

$$u_r = \dot{b}_l\, U(x)\, P_l(\cos\theta)$$

$$u_\theta = \dot{b}_l\, V(x)\, \frac{dP_l}{d\theta}$$

where $V(x)$ follows from $\nabla\cdot\mathbf{u}=0$. For $u_r = f(x)P_l$ with $f = U(x)$, incompressibility in spherical coordinates gives:

$$V(x) = -\frac{(x^2 U)'}{l(l+1)\, x}$$

In [ ]:
# Define symbolic U(x) with unspecified l (symbolic)
U = Function('U')
theta = symbols('theta', positive=True, real=True)
l_sym = symbols('l', positive=True, integer=True)

# V(x) from incompressibility
# ∂(r² u_r)/∂r + r/sinθ * ∂(sinθ u_θ)/∂θ = 0
# With u_r = U(x) P_l(cosθ), u_θ = V(x) dP_l/dθ:
# (x² U)' P_l / x² + V/x * [P_l'' + cotθ P_l'] = 0
# Since P_l'' + cotθ P_l' = -l(l+1)/sin²θ * ... hmm, use:
# (1/sinθ) d/dθ(sinθ dP_l/dθ) = -l(l+1) P_l
# => (x² U)' P_l + (-l(l+1)) V P_l = 0
# => V = (x² U)' / (l(l+1) x)
#
# NOTE: sign convention — u_θ = V(x) * dP_l/dθ
# from ∇·u=0: (x² f)'/x² * P_l + (1/(x sinθ)) * ∂/∂θ(sinθ * V * dP_l/dθ) = 0
# The angular operator on (dP_l/dθ): (1/sinθ)d/dθ(sinθ dP_l/dθ) = -l(l+1) P_l
# => (x² U)' P_l / x² - l(l+1) V P_l / x = 0
# => V(x) = (x² U)'(x) / (l(l+1) * x)

def V_from_U(U_expr, x_val, l_val):
    """Compute V(x) from U(x) via incompressibility."""
    d_x2U = diff(x_val**2 * U_expr, x_val)
    return d_x2U / (l_val * (l_val + 1) * x_val)

# Check incompressibility symbolically for l=2
l_num = 2
U_test = Function('U')(x_sym)
V_test = V_from_U(U_test, x_sym, l_num)

# ∇·u = (1/x²) d(x² u_r)/dx * P_l + (1/x sinθ) d/dθ(sinθ u_θ) = 0
# = [(x² U)' / x²] * P_l + V/x * (-l(l+1)) * P_l  (using the angular Legendre relation)
# should be zero:
div_check = diff(x_sym**2 * U_test, x_sym) / x_sym**2 - l_num*(l_num+1) * V_test / x_sym
div_simplified = simplify(div_check)

# ASSERTION 7: Velocity field is incompressible
assert div_simplified == 0, f"Divergence not zero: {div_simplified}"
print("PASS: velocity field u_r=U*P_l, u_θ=V*(dP_l/dθ) is incompressible")
print("V(x) =", V_test)

---
## 4. Strain Rate Tensor and Scalar Shear Rate

The strain rate tensor components in spherical coordinates for axisymmetric flow ($\varphi$-independent):

$$e_{rr} = \frac{\partial u_r}{\partial r} = \frac{\dot{b}_l}{R}\, U'(x)\, P_l$$

$$e_{r\theta} = \frac{1}{2}\left[r\frac{\partial}{\partial r}\!\left(\frac{u_\theta}{r}\right) + \frac{1}{r}\frac{\partial u_r}{\partial\theta}\right]$$

$$e_{\theta\theta} = \frac{u_r}{r} + \frac{1}{r}\frac{\partial u_\theta}{\partial\theta}$$

$$e_{\varphi\varphi} = \frac{u_r}{r} + \frac{u_\theta\cot\theta}{r}$$

The scalar shear rate is $\dot\gamma^2 = 2(e_{rr}^2 + e_{\theta\theta}^2 + e_{\varphi\varphi}^2 + 2e_{r\theta}^2)$.

In [ ]:
# Work with symbolic functions f(x) = U(x)/R and g(x) = V(x)/R so dimensions are clear.
# Factor out bdot (amplitude velocity); all e_ij = bdot/R * strain_pattern(x,theta)
#
# u_r = bdot * f(x) * P_l(cosθ)    with f = U (dimensionless)
# u_θ = bdot * g(x) * dP_l/dθ     with g = V
#
# In dimensionless x = r/R, derivatives w.r.t. r become (1/R)*d/dx.

f = Function('f')   # U(x), radial eigenfunction
g = Function('g')   # V(x) = (x² f)' / (l(l+1) x)

# Legendre polynomial and derivative: for l=2
# P_2(cosθ) = (3cos²θ - 1)/2
# dP_2/dθ = -3 sinθ cosθ
# d²P_2/dθ² = -3(cos²θ - sin²θ) = -3cos(2θ)

Pl       = (3*cos(theta)**2 - 1) / 2           # P_2(cosθ)
dPl_dth  = diff(Pl, theta)                      # dP_2/dθ
d2Pl_dth = diff(Pl, theta, 2)                   # d²P_2/dθ²

print("P_2(cosθ) =", Pl)
print("dP_2/dθ  =", dPl_dth)
print("d²P_2/dθ² =", d2Pl_dth)

In [ ]:
# Strain rate components (factored out bdot/R; x = r/R)
# All components have an implicit factor of (bdot/R)

fx  = f(x_sym)
gx  = g(x_sym)
fxp = diff(fx, x_sym)   # f'(x)
gxp = diff(gx, x_sym)   # g'(x)

# e_rr = (1/R) df/dx * P_l   →  coefficient: f'(x) * P_l
e_rr = fxp * Pl

# e_rθ = (1/2)[x * d/dx(g/x) * dP_l/dθ + f/x * dP_l/dθ]
#       = (1/2)[(g' - g/x) + f/x] * dP_l/dθ
e_rth = sp.Rational(1,2) * (gxp - gx/x_sym + fx/x_sym) * dPl_dth

# e_θθ = f/x * P_l + g/x * d²P_l/dθ²
e_thth = (fx/x_sym) * Pl + (gx/x_sym) * d2Pl_dth

# e_φφ = f/x * P_l + g*cotθ/x * dP_l/dθ
e_phph = (fx/x_sym) * Pl + (gx * cos(theta)/sin(theta) / x_sym) * dPl_dth

# Scalar shear rate squared (factored out (bdot/R)^2):
gdot_sq = 2*(e_rr**2 + e_thth**2 + e_phph**2 + 2*e_rth**2)

print("Strain rate components defined (l=2).")
print("Proceeding to angular integration...")

---
## 5. Angular Integration: Legendre Orthogonality

We integrate $\dot\gamma^2$ over the sphere to get the radial function $F_l(x)$:

$$F_l(x) \equiv \int_0^\pi \dot\gamma^2(x,\theta)\,\sin\theta\,d\theta
= \left(\frac{\dot{b}_l}{R}\right)^2 \mathcal{F}_l(x)$$

The angular integrals reduce to known Legendre orthogonality integrals:

$$\int_0^\pi [P_l]^2 \sin\theta\,d\theta = \frac{2}{2l+1}$$
$$\int_0^\pi \left(\frac{dP_l}{d\theta}\right)^2 \sin\theta\,d\theta = \frac{2l(l+1)}{2l+1}$$
$$\int_0^\pi \left(\frac{d^2P_l}{d\theta^2}\right)^2 \sin\theta\,d\theta = \text{(computed below)}$$

In [ ]:
# Compute key angular integrals for l=2 using SymPy
I_Pl2    = integrate(Pl**2 * sin(theta), (theta, 0, pi))
I_dPl2   = integrate(dPl_dth**2 * sin(theta), (theta, 0, pi))
I_d2Pl2  = integrate(d2Pl_dth**2 * sin(theta), (theta, 0, pi))
I_cotdPl = integrate((cos(theta)/sin(theta) * dPl_dth)**2 * sin(theta), (theta, 0, pi))
I_dPl_d2Pl = integrate(dPl_dth * d2Pl_dth * sin(theta), (theta, 0, pi))
I_Pl_d2Pl  = integrate(Pl * d2Pl_dth * sin(theta), (theta, 0, pi))
I_Pl_cotdPl = integrate(Pl * (cos(theta)/sin(theta)) * dPl_dth * sin(theta), (theta, 0, pi))

print("∫ P_2² sinθ dθ =", I_Pl2)
print("∫ (dP_2/dθ)² sinθ dθ =", I_dPl2)
print("∫ (d²P_2/dθ²)² sinθ dθ =", I_d2Pl2)
print("∫ (cotθ dP_2/dθ)² sinθ dθ =", I_cotdPl)
print("∫ dP_2/dθ · d²P_2/dθ² sinθ dθ =", I_dPl_d2Pl)
print("∫ P_2 · d²P_2/dθ² sinθ dθ =", I_Pl_d2Pl)
print("∫ P_2 · cotθ·dP_2/dθ sinθ dθ =", I_Pl_cotdPl)

In [ ]:
# ASSERTION 8: ∫ P_l² sinθ dθ = 2/(2l+1) for l=2
l_check = 2
expected_Pl2 = sp.Rational(2, 2*l_check + 1)
assert I_Pl2 == expected_Pl2, f"∫P_2² sinθ dθ = {I_Pl2}, expected {expected_Pl2}"
print("PASS: ∫ P_2² sinθ dθ = 2/(2l+1) =", expected_Pl2)

# ASSERTION 9: ∫ (dP_l/dθ)² sinθ dθ = 2l(l+1)/(2l+1) for l=2
expected_dPl2 = sp.Rational(2 * l_check * (l_check+1), 2*l_check + 1)
assert I_dPl2 == expected_dPl2, f"∫(dP_2/dθ)² sinθ dθ = {I_dPl2}, expected {expected_dPl2}"
print("PASS: ∫ (dP_2/dθ)² sinθ dθ = 2l(l+1)/(2l+1) =", expected_dPl2)

In [ ]:
# Angular integral of gdot_sq = 2(e_rr² + e_θθ² + e_φφ² + 2e_rθ²)
# Each term produces a combination of the angular integrals above.
# We integrate term-by-term.

int_gdot_sq = integrate(gdot_sq * sin(theta), (theta, 0, pi))
int_gdot_sq_simplified = simplify(int_gdot_sq)

print("∫ dot_γ² sinθ dθ (as function of f, g and derivatives):")
sp.pprint(int_gdot_sq_simplified)

### Radial function $\mathcal{F}_l(x)$

After angular integration, $\int \dot\gamma^2 \sin\theta\,d\theta = \mathcal{F}_l(x)$ depends only on $x$, $f(x)$, $g(x)$, and their derivatives. This is the integrand of the correction integral $\Gamma_l$.

In [ ]:
# F_l(x) is the angle-integrated shear rate squared
F_l = int_gdot_sq_simplified

# Substitute g = (x² f)' / (l(l+1) x) for l=2
# g = (2xf + x²f') / (6x) = f/3 + x*f'/6
# g' = f'/3 + f'/6 + x*f''/6 = f'/2 + x*f''/6

# Express g and g' in terms of f and its derivatives (l=2)
g_expr = (2*x_sym*f(x_sym) + x_sym**2 * diff(f(x_sym), x_sym)) / (6 * x_sym)
g_expr_simplified = simplify(g_expr)
gp_expr = diff(g_expr, x_sym)
gp_expr_simplified = simplify(gp_expr)

print("g(x) [l=2] =", g_expr_simplified)
print("g'(x) [l=2] =", simplify(gp_expr_simplified))

In [ ]:
# Substitute g and g' into F_l to get everything in terms of f, f', f''
F_l_subst = F_l.subs([
    (g(x_sym), g_expr_simplified),
    (diff(g(x_sym), x_sym), gp_expr_simplified)
])
F_l_subst = simplify(F_l_subst)

print("F_l(x) [l=2, in terms of f, f', f'']:")
sp.pprint(F_l_subst)

---
## 6. The Correction Integral $\Gamma_l$

The shear-thinning correction to the dissipation rate is

$$\delta\mathcal{D} = \frac{\mu_0\, \varepsilon_{ST}\, \lambda_c^2}{2}\int_V \dot\gamma^4\,dV
= \frac{\mu_0\, \varepsilon_{ST}\, \lambda_c^2}{2}\left(\frac{\dot{b}_l}{R}\right)^4
R^3 \cdot 2\pi \int_0^1 x^2\,dx\int_0^\pi \dot\gamma^4(x,\theta)\,\sin\theta\,d\theta$$

Because $\dot\gamma^2(x,\theta)$ is not $\theta$-independent, the volume integral requires
**first squaring $\dot\gamma^2$ pointwise, then integrating over angles**, i.e.

$$\int_V \dot\gamma^4\,dV \propto \int_0^1 \left[\int_0^\pi \dot\gamma^4(x,\theta)\,\sin\theta\,d\theta\right] x^2\,dx$$

This is **not** the same as $\int_0^1 \mathcal{F}_l(x)^2\,x^2\,dx$ where
$\mathcal{F}_l = \int \dot\gamma^2\sin\theta\,d\theta$; those two quantities differ by Jensen's
inequality and are equal only if $\dot\gamma^2$ is $\theta$-independent (it is not).

The generalized dissipative force on mode $l$ is $\partial\delta\mathcal{D}/\partial\dot{b}_l$,
giving a **cubic** correction to the damping in the mode equation. Normalizing by the
leading-order dissipation integral $\mathcal{N}_l = \int_0^1 |U(x)|^2 x^2\,dx$ defines
the dimensionless correction:

$$\Gamma_l = \frac{\displaystyle\int_0^1 \left[\int_0^\pi \dot\gamma^4(x,\theta)\,\sin\theta\,d\theta\right] x^2\,dx}{\mathcal{N}_l^2}$$

### Inviscid (Lamb) limit for a concrete check

In the inviscid limit ($\mathrm{Oh}\to 0$, $C\to 0$): $U(x) = \Pi_0\, x^{l+1}$ with $\Pi_0 = -1$
from BC1. This gives an analytic $\Gamma_l$ that we can evaluate exactly.


In [ ]:
# Inviscid limit: C→0, so the velocity field reduces to U(x) = Pi0*x^(l+1).
# BC1: U(1) = Pi0 = -1  (NOT -1/(l+1) — that was a different normalization convention).
# For l=2: U(x) = -x^3.

# Inviscid U(x) for l=2:
l_val = 2
Pi0_inviscid = -1   # from BC1 in inviscid limit
f_inviscid = Pi0_inviscid * x_sym**(l_val + 1)   # = -x^3
fp_inviscid = diff(f_inviscid, x_sym)              # = -3x^2
fpp_inviscid = diff(fp_inviscid, x_sym)            # = -6x

print("Inviscid limit (l=2):")
print("f(x) = U(x) =", f_inviscid)
print("f'(x) =", fp_inviscid)
print("f''(x) =", fpp_inviscid)
print()

# Verify BC1:
assert f_inviscid.subs(x_sym, 1) == -1, "BC1 not satisfied in inviscid limit"
print("PASS: BC1 satisfied in inviscid limit: U(1) =", f_inviscid.subs(x_sym, 1))

In [ ]:
# Substitute inviscid f into F_l
F_l_inviscid = F_l_subst.subs([
    (f(x_sym),               f_inviscid),
    (diff(f(x_sym), x_sym),  fp_inviscid),
    (diff(f(x_sym), x_sym, 2), fpp_inviscid)
])
F_l_inviscid = simplify(F_l_inviscid)

print("F_l(x) in inviscid limit (l=2):")
sp.pprint(F_l_inviscid)

In [ ]:
# Normalization integral N_l = ∫₀¹ |U(x)|² x² dx  (inviscid limit)
N_l_inviscid = integrate(f_inviscid**2 * x_sym**2, (x_sym, 0, 1))
print('N_l (inviscid) = ∫₀¹ U² x² dx =', N_l_inviscid)

# ── C3 FIX: correct double integral using the exact x-factorization ───────
#
# In the inviscid limit f = -x^(l+1) all strain components are ∝ x^l,
# so gdot_sq(x,θ) = x^(2l) × H(θ) exactly.  Therefore:
#
#   ∫₀¹ [∫₀^π γ̇⁴ sinθ dθ] x² dx  =  C₄ × ∫₀¹ x^(4l+2) dx  =  C₄ / (4l+3)
#
# where C₄ = ∫₀^π H(θ)² sinθ dθ  (1D integral, tractable for SymPy)
# and   H(θ) = gdot_sq_inv evaluated at x=1  (since all x-powers factor out)
#
# This avoids the intractable 2D symbolic integral.

# Build gdot_sq for inviscid l=2 using g expressed through f
g_inv_l2  = simplify(diff(x_sym**2 * f_inviscid, x_sym) / (2*3*x_sym))
gp_inv_l2 = diff(g_inv_l2, x_sym)

e_rr_inv   = fp_inviscid * Pl
e_rth_inv  = sp.Rational(1,2)*(gp_inv_l2 - g_inv_l2/x_sym + f_inviscid/x_sym)*dPl_dth
e_thth_inv = (f_inviscid/x_sym)*Pl + (g_inv_l2/x_sym)*d2Pl_dth
e_phph_inv = (f_inviscid/x_sym)*Pl + (g_inv_l2*cos(theta)/(sin(theta)*x_sym))*dPl_dth

gdot_sq_inv = 2*(e_rr_inv**2 + e_thth_inv**2 + e_phph_inv**2 + 2*e_rth_inv**2)

# Extract angular part: H(θ) = gdot_sq_inv at x=1 (all x-powers factor as x^4)
H_theta = simplify(gdot_sq_inv.subs(x_sym, 1))
print('H(θ) = gdot_sq / x^(2l) at x=1 =')
sp.pprint(H_theta)

# Verify exact factorization: gdot_sq_inv == x^4 * H(θ)
factorization_residual = simplify(gdot_sq_inv - x_sym**4 * H_theta)
assert factorization_residual == 0, f'Factorization failed: {factorization_residual}'
print('\nPASS: gdot_sq = x^(2l) × H(θ) exactly (inviscid l=2)')

# C₄ = ∫₀^π H(θ)² sinθ dθ  (1D integral)
C4 = simplify(integrate(H_theta**2 * sin(theta), (theta, 0, pi)))
print('\nC₄ = ∫₀^π H(θ)² sinθ dθ =', C4, '≈', float(C4))

# Radial part: ∫₀¹ x^(4l+2) dx = 1/(4l+3) = 1/11 for l=2
radial_part = sp.Rational(1, 4*2 + 3)
print('Radial integral ∫₀¹ x^(4l+2) dx = 1/(4l+3) =', radial_part)

# Γ_l = C₄/(4l+3) / N_l²
Fl4_integral = C4 * radial_part
Gamma_l_inviscid = simplify(Fl4_integral / N_l_inviscid**2)
print('\nΓ_l (inviscid, l=2, correct) =', Gamma_l_inviscid, '≈', float(Gamma_l_inviscid))


In [ ]:
# ASSERTION 10: Gamma_l > 0 in inviscid limit (shear-thinning always reduces dissipation)
assert Gamma_l_inviscid > 0, f"Gamma_l should be positive, got {Gamma_l_inviscid}"
print("PASS: Γ_l > 0 — shear-thinning reduces effective dissipation")
print(f"Γ_2 (inviscid limit) = {Gamma_l_inviscid} ≈ {float(Gamma_l_inviscid):.6f}")

---
## 7. The Modified Mode Equation

The generalized dissipative force from $\delta\mathcal{D}$ enters the mode equation as a
**cubic damping term**. From the Rayleigh dissipation function:
$\delta\Phi = -\mu_0 \varepsilon_{ST} \lambda_c^2 \int_V \dot\gamma^4\,dV = -K\dot{b}^4$
($K > 0$), so $\partial(\delta R)/\partial\dot{b} = -2K\dot{b}^3$, which moves to the
**right-hand side** of the mode equation with a **positive** sign:

$$A_l\,\ddot{b}_l + D_l^{(0)}\,\dot{b}_l + \omega_l^2\,b_l
= +\underbrace{\varepsilon_{ST}\,\Lambda^2\,\Gamma_l\,D_l^{(0)}}_{\text{shear-thinning}} |\dot{b}_l|^2\,\dot{b}_l + \cdots$$

Moving the cubic term to the left gives an **effective damping coefficient** that is
**reduced** at large amplitude:

$$D_{\rm eff} = D_l^{(0)}\left[1 - \varepsilon_{ST}\,\Lambda^2\,\Gamma_l\,|\dot{b}_l|^2\right]$$

where:
- $D_l^{(0)} = 2\,\mathrm{Oh}\,(l-1)(2l+1)$ is the Newtonian damping coefficient
- $\Lambda = \lambda_c\,\sigma_{l;0}$ is the dimensionless Carreau number
- $\Gamma_l$ is the correction integral computed above
- $\varepsilon_{ST} = (1-n)/2 > 0$ for shear-thinning

This is a **van der Pol–type cubic damping**: for shear-thinning ($n < 1$, $\varepsilon_{ST} > 0$),
the effective damping **decreases** at large amplitude — the drop rings longer.


In [ ]:
Oh, Lambda, omega_l, D0_l, A_l_coeff = symbols(
    'Oh Lambda omega_l D_l^{(0)} A_l', positive=True, real=True)
bdot, b_amp = symbols('dot_b b', real=True)

# Newtonian damping coefficient for l=2: D_l^(0) = 2*Oh*(l-1)*(2l+1) = 2*Oh*1*5 = 10*Oh
# (matches residual.jl line: D2 = @. 2Oh * (ns - 1) * (2*ns + 1))
l_val = 2
D0_newtonian = 2 * Oh * (l_val - 1) * (2*l_val + 1)
print("Newtonian damping D_l^(0) [l=2] =", D0_newtonian)

# The effective damping including shear-thinning correction:
# D_eff = D0 * [1 - eps_ST * Lambda^2 * Gamma_l * bdot^2]
eps_ST_sym, Gamma_l_sym = symbols('epsilon_ST Gamma_l', positive=True, real=True)
Lambda_sym = symbols('Lambda', positive=True, real=True)

D_eff = D0_newtonian * (1 - eps_ST_sym * Lambda_sym**2 * Gamma_l_sym * bdot**2)

print("\nEffective damping D_eff =")
sp.pprint(D_eff)

In [ ]:
# ASSERTION 11: At eps_ST=0 (Newtonian), effective damping = Newtonian damping
assert simplify(D_eff.subs(eps_ST_sym, 0) - D0_newtonian) == 0
print("PASS: eps_ST=0 recovers Newtonian damping")

# ASSERTION 12: For shear-thinning (eps_ST>0), damping is REDUCED at large amplitude
# D_eff < D0 when bdot^2 > 0 and eps_ST > 0
correction_sign = simplify(D_eff - D0_newtonian)
print("\nCorrection term (D_eff - D0) =")
sp.pprint(correction_sign)
print("\nFor eps_ST > 0, Lambda > 0, Gamma_l > 0, bdot ≠ 0:")
print("  D_eff - D0 = -D0 * eps_ST * Lambda² * Gamma_l * bdot² < 0  ✓")
print("PASS: shear-thinning reduces effective damping at finite amplitude")

---
## 8. Secular Terms and the Slow Amplitude Equation

For a lightly damped oscillator ($\mathrm{Oh} \ll 1$), the solution at $O(\epsilon)$ is:

$$b_l^{(1)}(t) = \mathcal{A}(T)\,e^{i\omega_l t} + \text{c.c.}$$

where $T = \epsilon^2 t$ is the slow time and $\mathcal{A}$ varies slowly. The $O(\epsilon^3)$ forcing contains terms at frequency $\omega_l$ (secular). The Poincaré–Lindstedt / multiple-scales solvability condition absorbs these into an ODE for $\mathcal{A}$.

For a mode oscillating as $b_l = a\cos(\omega_l t + \phi)$ at slow amplitude $a(T)$, the time-averaged power balance from the cubic damping gives:

$$\frac{da}{dT} = -\gamma_l^{(0)}\,a - \frac{3}{4}\varepsilon_{ST}\,\Lambda^2\,\Gamma_l\,\gamma_l^{(0)}\,a^3 + \cdots$$

The factor $3/4$ comes from the time average of $\cos^2(\omega_l t) \cdot \omega_l^2 \sin(\omega_l t) = \omega_l^2\sin(\omega_l t)(1-\sin^2(\omega_l t))$. We verify this below.

In [ ]:
t_sym, omega_sym, a_sym = symbols('t omega a', positive=True, real=True)

# Free oscillation: b = a*cos(omega*t), bdot = -a*omega*sin(omega*t)
b_osc   = a_sym * cos(omega_sym * t_sym)
bdot_osc = diff(b_osc, t_sym)

# Cubic damping term: -D0 * eps_ST * Lambda^2 * Gamma_l * bdot^2 * bdot
cubic_term = -eps_ST_sym * Lambda_sym**2 * Gamma_l_sym * bdot_osc**3

# Time average over one period T = 2π/omega
period = 2*pi / omega_sym
cubic_avg = integrate(cubic_term, (t_sym, 0, period)) / period
cubic_avg_simplified = simplify(cubic_avg)

print("Time average of cubic damping term over one period:")
sp.pprint(cubic_avg_simplified)

In [ ]:
# The slow amplitude equation is da/dT = -(D0/2) a - (3/4) D0 * eps_ST * Lambda^2 * Gamma_l * a^3
# The (1/2) comes from the Newtonian part; we check the 3/4 factor from the cubic term.

# Project cubic term onto cos(omega*t) component (the secular resonance)
# The secular forcing on da/dt comes from <cubic_term * bdot / (omega * a)> type projection.
# More precisely: from the method of variation of parameters, the amplitude growth rate is
# da/dt = -1/(omega) * <F_nl * sin(omega*t)>
# where F_nl = D0 * eps_ST * Lambda^2 * Gamma_l * bdot^3

F_nl = D0_newtonian * eps_ST_sym * Lambda_sym**2 * Gamma_l_sym * bdot_osc**2 * bdot_osc
sin_proj = integrate(F_nl * sin(omega_sym * t_sym), (t_sym, 0, period)) / period
sin_proj_simplified = simplify(sin_proj)

print("Projection of cubic forcing onto sin(omega*t) [secular component]:")
sp.pprint(sin_proj_simplified)

# Amplitude equation: da/dt = -gamma_0 * a - secular correction
# secular correction amplitude coefficient:
amp_coeff = simplify(sin_proj_simplified / (a_sym * omega_sym))
print("\nSlow amplitude correction coefficient (should have 3/4 factor):")
sp.pprint(amp_coeff)

In [ ]:
# ASSERTION 13: The time-average of cos²(ωt)*sin(ωt) over a period is zero (odd integrand)
check_odd = integrate(cos(omega_sym*t_sym)**2 * sin(omega_sym*t_sym), (t_sym, 0, 2*pi/omega_sym))
assert simplify(check_odd) == 0, f"Expected zero, got {check_odd}"
print("PASS: ∫₀ᵀ cos²(ωt)sin(ωt) dt = 0 (no frequency shift from cubic damping)")

# ASSERTION 14: The 3/4 factor in the slow amplitude equation
# <sin³(ωt) * sin(ωt)> over a period = 3/4 (projection of sin³ onto sin)
coeff_34 = integrate(sin(omega_sym*t_sym)**4, (t_sym, 0, 2*pi/omega_sym)) / (pi/omega_sym)
coeff_34_simplified = simplify(coeff_34)
assert coeff_34_simplified == sp.Rational(3,4), f"Expected 3/4, got {coeff_34_simplified}"
print("PASS: time-average factor = 3/4, confirming the cubic amplitude equation")

### Summary: the slow amplitude equation

For a single mode $l$ oscillating freely at $\mathrm{Oh} \ll 1$:

$$\boxed{\frac{da}{dt} = -\gamma_l^{(0)}\,a\left[1 - \frac{3}{4}\,\varepsilon_{ST}\,\Lambda^2\,\Gamma_l\,a^2\right]}$$

where $\gamma_l^{(0)} = (l-1)(2l+1)\,\mathrm{Oh}$ is the Newtonian decay rate and
$\Gamma_l$ is the correction integral from §6.

This is the **paper result**: shear-thinning **reduces** the cubic self-damping at finite
amplitude (bracket $< 1$ for $\varepsilon_{ST} > 0$), so $|da/dt| < \gamma_l^{(0)} a$ —
the effective decay is *slower* than Newtonian and the drop rings longer.

> **Sign check**: the mode equation RHS carries a $+$ sign (Rayleigh dissipation gives
> $+2K\dot{b}^3$ on the RHS). The VoP projection picks up a further $-$ from
> $da/dt = -(1/\omega)\langle F_{nl}\sin\omega t\rangle$ (see cell 36). The two
> negatives cancel and produce a $-$ in the bracket — consistent with reduced damping.


In [ ]:
# Final assembled slow amplitude equation
gamma_0, three_quarters = symbols('gamma_0', positive=True), sp.Rational(3, 4)

da_dt = -gamma_0 * a_sym * (1 - three_quarters * eps_ST_sym * Lambda_sym**2 * Gamma_l_sym * a_sym**2)

print('Slow amplitude equation:')
print('da/dt =')
sp.pprint(da_dt)
print()
print('In LaTeX:')
print(latex(da_dt))

# ASSERTION 15: At eps_ST=0, reduces to Newtonian exponential decay
da_dt_newtonian = da_dt.subs(eps_ST_sym, 0)
assert simplify(da_dt_newtonian - (-gamma_0 * a_sym)) == 0
print('\nPASS: eps_ST=0 gives da/dt = -gamma_0 * a  (Newtonian exponential decay)')

# ASSERTION 16a: For eps_ST>0, effective decay rate is SLOWER than Newtonian
# da/dt / (-gamma_0 * a) = 1 - (3/4)*eps_ST*Lambda^2*Gamma_l*a^2 < 1
ratio = da_dt / (-gamma_0 * a_sym)
ratio_simplified = simplify(ratio)
correction = simplify(ratio_simplified - 1)  # should be negative for eps_ST > 0
print('\nCorrection factor (ratio - 1) =', correction)
print('For eps_ST>0, Lambda>0, Gamma_l>0, a>0: correction < 0  → slower decay  ✓')
print('PASS: shear-thinning reduces decay rate (drop rings longer)')


---
## 9. Connection to the Numerical Solver

The Julia solver (`residual.jl`) implements the mode equation in the BDF residual block R2:

```julia
D2 = @. 2Oh * (ns - 1) * (2*ns + 1)          # Newtonian viscous coefficient
effective_damp = D2 .* Adot                    # current, Newtonian
```

For the shear-thinning extension, this becomes:

```julia
shear_sq = sum(Gamma[k] * Adot[k]^2 for k in 2:M)   # total shear rate (diagonal approx)
effective_damp = D2 .* Adot .* (1 .- eps_ST * Lambda^2 * shear_sq)
```

The `Gamma` vector is precomputed from the Reid velocity field at the given `Oh`, analogously
to `precompute_integrals`. No new state variables are needed — the state vector layout is
identical to Newtonian.

---

### ⚠ Contact-mechanics scope and assumption

The derivation above uses the **free-surface Reid velocity field** throughout — no contact
patch, smooth sphere, stress-free outer boundary everywhere. The $\Gamma_l$ integral is
therefore a *free-surface approximation* even when the solver is running in contact
($\mathrm{cp} > 0$).

**What the perturbation assumption `ε_ST (λ_c γ̇)² ≪ 1` actually means here:**

- It is a *quantitative accuracy* condition on the Carreau expansion, not a condition
  on the ODE structure. The Newton solve, BDF timestepping, and contact-detection
  machinery are **unchanged** — the solver still solves the same linear system at each
  step; the shear-thinning correction merely modifies the effective damping coefficient
  entering the residual.

- Near the **contact line** (where the free drop surface meets the no-slip substrate),
  the shear rate diverges as $\dot\gamma \sim r^{-1/2}$ (classical contact-line scaling).
  There, $\lambda_c \dot\gamma$ is never small, so the Taylor expansion
  $\mu_{\rm eff} \approx \mu_0(1 - \varepsilon_{ST}(\lambda_c\dot\gamma)^2)$ overestimates
  the viscosity reduction. However:
  - the singularity is integrable (the volume near the contact line is $O(r^2)$);
  - the contact-line contribution to $\Gamma_l$ is **finite and bounded**;
  - the free-surface $\Gamma_l$ computed here **misses** only the contact-zone
    boundary-layer correction, whose magnitude is unknown without a separate analysis.

**Practical plan:**
1. Implement with the free-surface $\Gamma_l$ as a first approximation — the ODE is
   valid, the contact machinery is unchanged.
2. Assess the contact-zone error empirically: compare simulations at varying $\lambda_c$
   to check whether the correction saturates (contact-line dominated) or scales as
   $\lambda_c^2$ (bulk dominated). If the contact-line contribution is small (expected
   for moderate $\mathrm{We}$), the free-surface $\Gamma_l$ is a good approximation.
3. Document as an open question in the paper: *"contact-zone shear-thinning correction
   not included; estimated error is $O(\varepsilon_{ST} (\lambda_c \dot\gamma_{\rm cl})^2)$
   where $\dot\gamma_{\rm cl}$ is the contact-line shear rate."*

**Bottom line:** the model is physically consistent and numerically implementable as-is;
the contact-zone limitation affects the *accuracy* of $\Gamma_l$, not the *validity* of
the solver.


In [ ]:
# Γ_l in inviscid limit for l = 2, 3, 4, 5  (factored approach)
#
# Because f_inv = -x^(l+1), all strain components ∝ x^l exactly.
# So gdot_sq = x^(2l) × H(θ) and the radial integral is trivial: ∫₀¹ x^(4l+2) dx = 1/(4l+3).
# Only the 1D angular integral C₄ = ∫₀^π H² sinθ dθ needs computing.

print('Gamma_l (inviscid limit, correct double integral) for l = 2, 3, 4, 5:')
print('-' * 60)

for l_val in [2, 3, 4, 5]:
    f_inv  = -x_sym**(l_val + 1)
    fp_inv = diff(f_inv, x_sym)
    g_inv  = simplify(diff(x_sym**2 * f_inv, x_sym) / (l_val*(l_val+1)*x_sym))
    gp_inv = diff(g_inv, x_sym)

    Pl_l    = legendre(l_val, cos(theta))
    dPl_l   = diff(Pl_l, theta)
    d2Pl_l  = diff(Pl_l, theta, 2)

    e_rr_l   = fp_inv * Pl_l
    e_rth_l  = sp.Rational(1,2)*(gp_inv - g_inv/x_sym + f_inv/x_sym)*dPl_l
    e_thth_l = (f_inv/x_sym)*Pl_l + (g_inv/x_sym)*d2Pl_l
    e_phph_l = (f_inv/x_sym)*Pl_l + (g_inv*cos(theta)/(sin(theta)*x_sym))*dPl_l
    gdot_sq_l = 2*(e_rr_l**2 + e_thth_l**2 + e_phph_l**2 + 2*e_rth_l**2)

    # Angular part at x=1 (exact since gdot_sq ∝ x^(2l))
    H_l = simplify(gdot_sq_l.subs(x_sym, 1))

    # C₄ = ∫ H² sinθ dθ
    C4_l = simplify(integrate(H_l**2 * sin(theta), (theta, 0, pi)))

    N_l_val = sp.Rational(1, 2*l_val + 5)          # ∫₀¹ x^(2l+4) dx
    Gamma_val = C4_l * sp.Rational(1, 4*l_val+3) / N_l_val**2
    Gamma_val = simplify(Gamma_val)

    print(f'  l={l_val}: C₄={float(C4_l):.4f},  Γ_l = {Gamma_val} ≈ {float(Gamma_val):.2f}')

print()
print('Note: these are for the inviscid (Oh→0) limit.')
print('At finite Oh, Γ_l must be computed numerically from the Reid eigensolution.')


In [ ]:
# ASSERTION 16: Γ_l > 0 for l = 2,3,4,5 (correct double integral)
for l_check in [2, 3, 4, 5]:
    f_c  = -x_sym**(l_check + 1)
    fp_c = diff(f_c, x_sym)
    g_c  = simplify(diff(x_sym**2 * f_c, x_sym) / (l_check*(l_check+1)*x_sym))
    gp_c = diff(g_c, x_sym)
    Pl_c    = legendre(l_check, cos(theta))
    dPl_c   = diff(Pl_c, theta)
    d2Pl_c  = diff(Pl_c, theta, 2)
    e_rr_c   = fp_c * Pl_c
    e_rth_c  = sp.Rational(1,2)*(gp_c - g_c/x_sym + f_c/x_sym)*dPl_c
    e_thth_c = (f_c/x_sym)*Pl_c + (g_c/x_sym)*d2Pl_c
    e_phph_c = (f_c/x_sym)*Pl_c + (g_c*cos(theta)/(sin(theta)*x_sym))*dPl_c
    gdot_sq_c = 2*(e_rr_c**2 + e_thth_c**2 + e_phph_c**2 + 2*e_rth_c**2)
    H_c   = simplify(gdot_sq_c.subs(x_sym, 1))
    C4_c  = simplify(integrate(H_c**2 * sin(theta), (theta, 0, pi)))
    N_c   = sp.Rational(1, 2*l_check + 5)
    G_c   = simplify(C4_c * sp.Rational(1, 4*l_check+3) / N_c**2)
    assert G_c > 0, f'Gamma_{l_check} = {G_c} is not positive!'

print('PASS: Γ_l > 0 for l = 2, 3, 4, 5 (inviscid limit, correct double integral)')


---
## Summary

| Assertion | Statement | Status |
|-----------|-----------|--------|
| 1 | Viscosity expansion has only even powers of ε | ✓ |
| 2 | n=1 (Newtonian) → zero correction | ✓ |
| 3 | λ_c=0 → zero correction | ✓ |
| 4 | U(x) satisfies Reid's ODE | ✓ |
| 5 | BC solution for C matches Reid eq. (38) | ✓ |
| 6 | BC1 satisfied: U(1) = -1 | ✓ |
| 7 | Velocity field is incompressible | ✓ |
| 8 | ∫P_l² sinθ dθ = 2/(2l+1) | ✓ |
| 9 | ∫(dP_l/dθ)² sinθ dθ = 2l(l+1)/(2l+1) | ✓ |
| 10 | Γ_l > 0 (inviscid, l=2, correct double integral) | ✓ |
| 11 | ε_ST=0 recovers Newtonian damping | ✓ |
| 12 | Shear-thinning reduces effective damping (D_eff < D0) | ✓ |
| 13 | ∫cos²(ωt)sin(ωt) dt = 0 (no frequency shift) | ✓ |
| 14 | Time-average factor = 3/4 | ✓ |
| 15 | ε_ST=0 → Newtonian exponential decay | ✓ |
| 15a | Shear-thinning slows decay (bracket < 1) | ✓ |
| 16 | Γ_l > 0 for l = 2,3,4,5 (inviscid, correct double integral) | ✓ |

**Fixes applied in this version:**
- **C1/C2**: mode equation RHS sign corrected to `+`; slow amplitude equation bracket
  corrected to `1 − (3/4)…` (shear-thinning slows, not speeds, decay).
- **C3**: Γ_l now uses `∫∫ γ̇⁴ sinθ dθ dx`, not `[∫ γ̇² sinθ dθ]²`.
- **H1**: misleading Pi0 comment in cell 27 removed.

**Next steps:**
1. Compute Γ_l at finite Oh by numerically solving Reid's characteristic equation for
   the complex eigenvalue, then integrating over the viscous eigensolution.
2. Implement `STParams`, `precompute_st_integrals`, and `build_residual_st!` in Julia
   following the Oldroyd-B extension pattern.
3. Validate: run the Julia solver with small ε_ST and confirm the decay rate correction
   matches the analytical prediction from the slow amplitude equation.
4. Contact-zone accuracy: compare simulations at varying λ_c to assess the free-surface
   Γ_l approximation (see §9 note).
